# Route Factory

> Route factory for job trigger, SSE progress streaming, and cancellation.

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
import asyncio
from typing import Any, Callable, Dict, List, Optional, Tuple
from dataclasses import asdict

from fasthtml.common import Div, Span, Script, APIRouter, FT, EventStream, sse_message

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot
from cjm_fasthtml_job_monitor.services.monitor import JobMonitorService
from cjm_fasthtml_job_monitor.components.trigger import render_job_trigger, render_job_progress_button
from cjm_fasthtml_job_monitor.components.overlay import render_job_overlay, render_job_overlay_placeholder
from cjm_fasthtml_job_monitor.components.modal import (
    render_job_modal, render_sse_connection, render_sse_response,
)

## init_job_monitor_routes

Factory function that creates the three routes (trigger, progress SSE stream, cancel)
and returns them as an `APIRouter` plus a `JobMonitorUrls` for consumers to reference.

### Consumer Setup

The consumer must:
1. Place `render_job_trigger(config, ids, urls)` in their toolbar/action area
2. Place `render_job_overlay_placeholder(ids)` inside the content area to overlay
   (the content area container must have `position: relative`)
3. Place an empty `Div(id=ids.modal)` as a modal placeholder at page level
4. Add `get_sse_headers()` to their app's header list (after HTMX)
5. Provide callbacks for job lifecycle events

In [ ]:
#| export
def _get_job_data(service, job_id):
    """Extract job status fields from a Job object."""
    job = service.get_job(job_id)
    if not job:
        return None, 'pending', 0.0, '', None, None
    return (
        job,
        job.status.value if hasattr(job.status, 'value') else str(job.status),
        job.progress or 0.0,
        job.status_message or '',
        job.started_at,
        job.completed_at,
    )

_TERMINAL_STATUSES = {'completed', 'failed', 'cancelled'}

In [ ]:
#| export
def init_job_monitor_routes(
    monitor_service: JobMonitorService,           # Service instance
    plugin_name: str,                             # Target plugin for jobs
    state_store: SQLiteWorkflowStateStore,         # For persisting job_id
    workflow_id: str,                             # Workflow ID for state access
    step_id: str,                                 # Step ID for state access
    state_key: str,                               # Key in step state for job_id
    prefix: str,                                  # URL prefix
    overlay_target_id: str,                       # ID of element to overlay
    kb_system_id: Optional[str] = None,           # Keyboard system ID to pause/resume
    on_complete: Optional[Callable] = None,       # async fn(job, request, sess) -> List[FT]
    on_cancel: Optional[Callable] = None,         # async fn(job, request, sess) -> List[FT]
    on_fail: Optional[Callable] = None,           # async fn(job, request, sess) -> List[FT]
    job_args_builder: Optional[Callable] = None,  # fn(state_store, workflow_id, session_id) -> (args, kwargs)
    config: Optional[JobMonitorConfig] = None,    # UI config
    id_prefix: str = "jm",                        # HTML ID prefix
    icon_fn: Optional[Callable] = None,           # Icon renderer fn(name, **kwargs) -> FT
) -> Tuple[APIRouter, JobMonitorUrls, JobMonitorHtmlIds]:  # (router, urls, ids)
    """Initialize job monitor routes with SSE-based progress streaming."""
    config = config or JobMonitorConfig()
    ids = JobMonitorHtmlIds(prefix=id_prefix)

    router = APIRouter(prefix=prefix)

    @router
    async def trigger(request, sess):
        """Submit a job and return the modal + overlay + progress button via OOB."""
        session_id = get_session_id(sess)

        # Build job args from consumer callback
        args, kwargs = (), {}
        if job_args_builder:
            args, kwargs = job_args_builder(state_store, workflow_id, session_id)

        # Submit to queue
        job_id = await monitor_service.submit_job(plugin_name, *args, **kwargs)

        # Persist job_id in step state
        state = state_store.get_state(workflow_id, session_id)
        step_states = state.get("step_states", {})
        step_state = step_states.get(step_id, {})
        step_state[state_key] = job_id
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)

        # Build URLs for SSE/cancel
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        # Get initial job data
        _, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)
        logs = monitor_service.get_logs(plugin_name, config.log_lines)
        resources = monitor_service.get_resource_snapshot(plugin_name)

        # Build OOB responses
        oob_parts = []

        # 1. Progress button replaces trigger slot
        prog_btn = render_job_progress_button(config, ids)
        prog_btn.attrs['hx-swap-oob'] = "true"
        oob_parts.append(prog_btn)

        # 2. Overlay replaces overlay placeholder
        overlay = render_job_overlay(ids, config)
        overlay.attrs['hx-swap-oob'] = "true"
        oob_parts.append(overlay)

        # 3. Modal with SSE connection (job_id passed to SSE endpoint)
        modal_el = render_job_modal(
            config, ids, urls, job_id=job_id,
            status=status, progress_value=prog_val,
            status_message=msg, started_at=started, completed_at=completed,
            logs=logs, resources=resources, open_on_render=True,
        )
        modal_el.attrs['hx-swap-oob'] = "true"
        oob_parts.append(modal_el)

        # 4. Keyboard pause script
        if kb_system_id:
            oob_parts.append(Script(f"window.kbCoordinator?.pause('{kb_system_id}');"))

        return tuple(oob_parts)

    @router
    async def progress(request, sess, job_id: str = ''):
        """SSE stream endpoint — pushes progress updates until job completes."""
        session_id = get_session_id(sess)

        # Build URLs
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        async def sse_stream():
            try:
                prev_logs = None
                resource_cycle = 0

                while True:
                    # Get current job state
                    job, status, prog_val, msg, started, completed = _get_job_data(
                        monitor_service, job_id
                    )

                    if not job:
                        yield sse_message(render_sse_response(ids, urls))
                        return

                    # Get logs (every cycle) and resources (every few cycles)
                    logs = monitor_service.get_logs(plugin_name, config.log_lines)
                    resources = None
                    if resource_cycle == 0:
                        resources = monitor_service.get_resource_snapshot(plugin_name)
                    resource_cycle = (resource_cycle + 1) % 4  # Resources every ~2s at 0.5s interval

                    # Check for terminal state
                    if status in _TERMINAL_STATUSES:
                        # Clear job_id from state
                        state = state_store.get_state(workflow_id, session_id)
                        step_states = state.get("step_states", {})
                        step_state = step_states.get(step_id, {})
                        step_state.pop(state_key, None)
                        step_states[step_id] = step_state
                        state["step_states"] = step_states
                        state_store.update_state(workflow_id, session_id, state)

                        # Get final resources snapshot
                        resources = monitor_service.get_resource_snapshot(plugin_name)

                        # Fire callback
                        extra_oob = []
                        if status == 'completed' and on_complete:
                            result = await on_complete(job, request, sess)
                            if result:
                                extra_oob.extend(
                                    result if isinstance(result, (list, tuple)) else [result]
                                )
                        elif status == 'cancelled' and on_cancel:
                            result = await on_cancel(job, request, sess)
                            if result:
                                extra_oob.extend(
                                    result if isinstance(result, (list, tuple)) else [result]
                                )
                        elif status == 'failed' and on_fail:
                            result = await on_fail(job, request, sess)
                            if result:
                                extra_oob.extend(
                                    result if isinstance(result, (list, tuple)) else [result]
                                )

                        # Cleanup OOB: remove overlay
                        overlay_placeholder = render_job_overlay_placeholder(ids)
                        overlay_placeholder.attrs['hx-swap-oob'] = "true"
                        extra_oob.append(overlay_placeholder)

                        # Restore trigger button
                        trigger_btn = render_job_trigger(config, ids, urls, icon_fn=icon_fn)
                        trigger_btn.attrs['hx-swap-oob'] = "true"
                        extra_oob.append(trigger_btn)

                        # Kill SSE connection — swap to inert element so HTMX
                        # doesn't auto-reconnect after the stream closes
                        inert_sse = render_sse_connection(ids, urls, job_id='', is_active=False)
                        inert_sse.attrs['hx-swap-oob'] = "true"
                        extra_oob.append(inert_sse)

                        # Resume keyboard
                        if kb_system_id:
                            extra_oob.append(
                                Script(f"window.kbCoordinator?.resume('{kb_system_id}');")
                            )

                        # Final SSE push with cleanup (include footer to remove Cancel)
                        yield sse_message(render_sse_response(
                            ids, urls, status=status, progress_value=prog_val,
                            status_message=msg, started_at=started,
                            completed_at=completed, logs=logs, resources=resources,
                            include_footer=True, extra_oob=extra_oob,
                        ))
                        return  # Close the stream

                    # Still running — push selective update
                    yield sse_message(render_sse_response(
                        ids, urls, status=status, progress_value=prog_val,
                        status_message=msg, started_at=started,
                        completed_at=completed,
                        logs=logs if logs != prev_logs else None,
                        resources=resources,
                    ))
                    prev_logs = logs

                    await asyncio.sleep(config.sse_interval_s)

            except Exception as e:
                print(f"[JobMonitor SSE] Error in stream: {e}")
                import traceback
                traceback.print_exc()
                return

        return EventStream(sse_stream())

    @router
    async def cancel(request, sess):
        """Cancel the active job."""
        session_id = get_session_id(sess)

        state = state_store.get_state(workflow_id, session_id)
        step_states = state.get("step_states", {})
        step_state = step_states.get(step_id, {})
        job_id = step_state.get(state_key)

        if job_id:
            await monitor_service.cancel_job(job_id)

        # Cancellation is async — the SSE stream detects the terminal state
        return ""

    # Build URLs
    urls = JobMonitorUrls(
        trigger=trigger.to(),
        progress=progress.to(),
        cancel=cancel.to(),
    )

    return router, urls, ids

## Resume Helper

Utility for consumers to check for in-flight jobs on page load.
Returns appropriate UI state (trigger button vs progress button, overlay vs placeholder).

In [ ]:
#| export
def check_inflight_job(
    monitor_service: JobMonitorService,       # Service instance
    plugin_name: str,                         # Target plugin
    state_store: SQLiteWorkflowStateStore,    # State store
    workflow_id: str,                         # Workflow ID
    session_id: str,                          # Session ID
    step_id: str,                             # Step ID
    state_key: str,                           # State key for job_id
    config: JobMonitorConfig,                 # Display config
    ids: JobMonitorHtmlIds,                   # Element IDs
    urls: JobMonitorUrls,                     # Route URLs
    icon_fn: Optional[Callable] = None,       # Icon renderer
) -> Tuple[Optional[FT], Optional[FT], Optional[FT], bool]:
    """Check for in-flight job and return appropriate UI state.
    
    Returns:
        - Button element (trigger or progress button)
        - Overlay element (active overlay or empty placeholder)
        - Modal element (with SSE connection if running, or empty placeholder)
        - Whether a job is currently running
    """
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    step_state = step_states.get(step_id, {})
    job_id = step_state.get(state_key)

    modal_placeholder = Div(id=ids.modal)

    if not job_id:
        # No in-flight job
        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            modal_placeholder,
            False,
        )

    # Check job status
    job, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)

    if not job or status in _TERMINAL_STATUSES:
        # Job finished -- clean up state
        step_state.pop(state_key, None)
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)

        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            modal_placeholder,
            False,
        )

    # Job is still running — return modal with SSE connection for resume
    logs = monitor_service.get_logs(plugin_name, config.log_lines)
    resources = monitor_service.get_resource_snapshot(plugin_name)

    modal_el = render_job_modal(
        config, ids, urls, job_id=job_id,
        status=status, progress_value=prog_val,
        status_message=msg, started_at=started, completed_at=completed,
        logs=logs, resources=resources,
        open_on_render=False,  # Don't auto-open on page load
    )

    return (
        render_job_progress_button(config, ids),
        render_job_overlay(ids, config),
        modal_el,
        True,
    )

In [ ]:
# Test URL generation
from cjm_fasthtml_job_monitor.models import JobMonitorUrls

# Verify URLs are constructed correctly
urls = JobMonitorUrls(
    trigger="/workflow/seg/fa/trigger",
    progress="/workflow/seg/fa/progress",
    cancel="/workflow/seg/fa/cancel",
)
assert urls.trigger.endswith("/trigger")
assert urls.progress.endswith("/progress")
assert urls.cancel.endswith("/cancel")
print(f"URLs: {urls}")
print("Route factory URL construction: OK")

In [ ]:
# Test check_inflight_job with no active job
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorConfig

class MockStateStore:
    def __init__(self, state=None):
        self._state = state or {}
    def get_state(self, wf_id, sess_id):
        return dict(self._state)
    def update_state(self, wf_id, sess_id, state):
        self._state = state

ids = JobMonitorHtmlIds(prefix="jm")
config = JobMonitorConfig()
urls = JobMonitorUrls(trigger="/trigger", progress="/progress", cancel="/cancel")
store = MockStateStore({"step_states": {"seg": {}}})

btn_el, overlay_el, modal_el, is_running = check_inflight_job(
    monitor_service=None,  # Not needed when no job_id in state
    plugin_name="test-plugin",
    state_store=store,
    workflow_id="wf1",
    session_id="s1",
    step_id="seg",
    state_key="fa_job_id",
    config=config,
    ids=ids,
    urls=urls,
)
assert not is_running
assert btn_el.attrs['id'] == 'jm-trigger-slot'
assert overlay_el.attrs['id'] == 'jm-overlay'
assert len(overlay_el.children) == 0  # Placeholder
assert modal_el.attrs['id'] == 'jm-modal'
assert modal_el.tag == 'div'  # Empty placeholder, not dialog
print("check_inflight_job (no active job): OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()